In [1]:
import pandas as pd
import numpy as np

out = './mbdump_small/'

# Load generated listening and relationship tables.
history       = pd.read_csv(f'{out}listening_history.tsv', sep='\t')
users         = pd.read_csv(f'{out}users.tsv',             sep='\t')
friends       = pd.read_csv(f'{out}friends.tsv',           sep='\t')
fav_artists   = pd.read_csv(f'{out}fav_artists.tsv',       sep='\t')

history['timestamp'] = pd.to_datetime(history['timestamp'])

# 1. Build implicit feedback features for ALS.
implicit = (history
    .groupby(['user_id','recording_id'])
    .agg(plays=('recording_id','count'),
         avg_duration=('duration_ms','mean'),
         completion_rate=('completed','mean'))
    .reset_index())

# 2. Build user/time-of-day play profiles for playlists.
tod_profile = (history
    .groupby(['user_id','recording_id','time_of_day'])
    .size().reset_index(name='play_count'))

# 3. Build per-user sessions for sequence-based experiments.
sessions = (history
    .sort_values('timestamp')
    .groupby(['user_id', pd.Grouper(key='timestamp', freq='30min')])['recording_id']
    .apply(list).reset_index())
sessions.columns = ['user_id', 'session_start', 'recording_sequence']
sessions = sessions[sessions['recording_sequence'].map(len) > 1]

# 4. Aggregate friends' listening activity for social recommendations.
friend_history = (friends
    .merge(history, left_on='friend_id', right_on='user_id', suffixes=('','_friend'))
    .groupby(['user_id','recording_id'])
    .agg(friend_plays=('recording_id','count'))
    .reset_index())

# 5. Compute weighted implicit scores from plays, completion, and duration.
implicit['implicit_score'] = (
    np.log1p(implicit['plays']) * 3.0 +
    implicit['completion_rate'] * 2.0 +
    (implicit['avg_duration'] / 300000).clip(0, 1) * 1.0
)

# Save feature tables as TSV files for Hive ingestion and model training.
feat = f'{out}features/'
import os
os.makedirs(feat, exist_ok=True)

implicit.to_csv(      f'{feat}implicit.tsv',       sep='\t', index=False)
tod_profile.to_csv(   f'{feat}tod_profile.tsv',    sep='\t', index=False)
sessions.to_csv(      f'{feat}sessions.tsv',        sep='\t', index=False)
friend_history.to_csv(f'{feat}friend_history.tsv', sep='\t', index=False)

# Print output sizes for ingestion checks.
for name, df in [('implicit', implicit), ('tod_profile', tod_profile),
                 ('sessions', sessions), ('friend_history', friend_history)]:
    print(f"{name:25s} → {len(df)} rows")

implicit                  → 90797 rows
tod_profile               → 137224 rows
sessions                  → 115614 rows
friend_history            → 960016 rows


In [2]:
# Operational notes for loading TSV outputs into HDFS.
# hdfs dfs -mkdir /aulas/francisco_jose_simoes/project/data
# hdfs dfs -put *.tsv /aulas/francisco_jose_simoes/project/data
# Each Hive external table expects one HDFS directory per source table.
# for f in artist artist_credit artist_credit_name artist_tag fav_artists friends l_recording_recording listening_history listening_history_real notifications recording recording_tag release release_group streaming_events tag track users; do
#   hdfs dfs -mkdir /aulas/francisco_jose_simoes/project/data/$f
#   hdfs dfs -mv /aulas/francisco_jose_simoes/project/data/$f.tsv /aulas/francisco_jose_simoes/project/data/$f/
# done